<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4I0_Nested_SCV_37_Record_Exploratory_Materialization_FINAL_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# STAGE 6C STEP 4I — CELL 6C-4I0
# 37-RECORD NESTED-SCV EXPLORATORY RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================
#
# Purpose
# -------
# 1. Freshly verify the immutable Stage 6B evaluable cohort, the frozen score-blind nested-SCV
#    record-level package, its manifest, and the completed Cell 6C-4H0 manifest.
# 2. Confirm that the high-rigor contradictory-submission endpoint remains non-estimable.
# 3. Isolate only the frozen 37 complete-case submitter-distribution records:
#       21 major-shift events and 16 negatives; 66,599 records censored.
# 4. Join the nine already-frozen Stage 6B scores only after the nested outcome package passes.
# 5. Reproduce the exact Cell 6C-3G1 point estimates and its 2,000 paired ordinary row-bootstrap
#    attempts using NumPy Generator(PCG64), seed 42, and rng.integers().
# 6. Materialize versioned score, replicate, interval, paired-comparison, historical-concordance,
#    QC, manifest, and SHA-256 sidecar artifacts.
#
# Scientific boundary
# -------------------
# - This is a highly exploratory 37-record complete-case sensitivity analysis.
# - The high-rigor endpoint has zero positive events and is not analyzed.
# - No score, outcome, model, threshold, weight, policy, linkage decision, row order, or cohort
#   membership is changed.
# - Experiment 2 is not started.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, EXPECTATIONS, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4I0_Nested_SCV_37_Record_"
    "Exploratory_Materialization.ipynb"
)

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
STAGE6_DIR = ROOT / "data_processed/stage6_temporal_validation"

EVAL = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVAL_SHA256 = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

NESTED_DIR = STAGE6_DIR / "nested_scv_secondary_outcomes"
NESTED_PACKAGE = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
NESTED_PACKAGE_SHA256 = "18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2"

NESTED_MANIFEST = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
NESTED_MANIFEST_SHA256 = "b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1/"
    "stage6c_4h0_alternative_outcome_secondary_drift_manifest_v1.json"
)
PRIOR_MANIFEST_SHA256 = "207eeb09c9f5d6d0f1e46058252b27c4f4906d26f918a08bb29780b7de00d2db"

EXPECTED_POLICY_SHA256 = "0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa"

EXPECTED = {
    "total_rows": 66_636,
    "eval_columns": 79,
    "nested_columns": 28,
    "complete_rows": 37,
    "events": 21,
    "negatives": 16,
    "censored": 66_599,
    "prevalence": 21 / 37,
    "censoring_fraction": 66_599 / 66_636,
    "high_rigor_events": 0,
    "high_rigor_negatives": 54_228,
    "high_rigor_censored": 12_408,
}

SEED = 42
N_BOOT = 2_000
KEYS = ["t0_row_order", "rcv_accession"]

SCORES = OrderedDict([
    ("full_ges", ("full_ges_instability_risk_t0", "Full GES")),
    ("no_star_ges", ("no_star_ges_instability_risk_t0", "No-star GES")),
    ("review_stars", ("review_stars_instability_risk", "Review stars")),
    ("combined_metadata", ("combined_metadata_instability_risk", "Combined metadata")),
    ("conflict", ("conflict_instability_risk", "Conflict")),
    ("recency", ("recency_instability_risk", "Recency")),
    ("submitter", ("submitter_instability_risk", "Submitter support")),
    ("entropy", ("entropy_instability_risk", "Classification entropy")),
    ("additive", ("additive_instability_risk", "Additive risk")),
])
PRINCIPAL_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)

for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "endpoint_accounting": TABLE_DIR / "stage6c_nested_scv_endpoint_accounting_v1.csv",
    "analysis_cohort": TABLE_DIR / "stage6c_nested_scv_37_record_analysis_cohort_v1.parquet",
    "point_estimates": TABLE_DIR / "stage6c_nested_scv_37_record_point_estimates_v1.csv",
    "bootstrap_replicates": TABLE_DIR / "stage6c_nested_scv_37_record_bootstrap_replicates_v1.parquet",
    "model_intervals": TABLE_DIR / "stage6c_nested_scv_37_record_model_bootstrap_intervals_v1.csv",
    "paired_inference": TABLE_DIR / "stage6c_nested_scv_37_record_paired_exploratory_inference_v1.csv",
    "historical_results": TABLE_DIR / "stage6c_nested_scv_37_record_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_nested_scv_37_record_historical_vs_reproduced_concordance_v1.csv",
    "limitations": TABLE_DIR / "stage6c_nested_scv_37_record_limitations_v1.csv",
    "qc": QC_DIR / "stage6c_4i0_nested_scv_37_record_exploratory_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4i0_nested_scv_37_record_exploratory_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Write atomically; on rerun, accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)

    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")

    if path.exists():
        existing = pd.read_parquet(path)
        fresh = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(
            existing,
            fresh,
            check_dtype=True,
            check_exact=True,
            check_like=False,
        )
        temporary.unlink()
    else:
        os.replace(temporary, path)

    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_hash(path: Path) -> str:
    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        Path(path).read_text(encoding="utf-8"),
    )
    if not matches:
        raise RuntimeError(f"No SHA-256 found in sidecar: {path}")
    return matches[0].lower()


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and sidecar_hash(output) == sha(path)


def normalize_rcv(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def interval_status(lower: float, upper: float) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0:
        return "full_ges_supported_higher"
    if upper < 0:
        return "full_ges_supported_lower"
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL PREFLIGHT
# --------------------------------------------------------------------------------------------------

required_files = [
    EVAL,
    EVAL.with_name(EVAL.name + ".sha256"),
    NESTED_PACKAGE,
    NESTED_PACKAGE.with_name(NESTED_PACKAGE.name + ".sha256"),
    NESTED_MANIFEST,
    NESTED_MANIFEST.with_name(NESTED_MANIFEST.name + ".sha256"),
    PRIOR_MANIFEST,
    PRIOR_MANIFEST.with_name(PRIOR_MANIFEST.name + ".sha256"),
]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(path)

if sha(EVAL) != EVAL_SHA256 or not sidecar_ok(EVAL):
    raise RuntimeError("Stage 6B evaluable package verification failed.")
if sha(NESTED_PACKAGE) != NESTED_PACKAGE_SHA256 or not sidecar_ok(NESTED_PACKAGE):
    raise RuntimeError("Nested-SCV record-level package verification failed.")
if sha(NESTED_MANIFEST) != NESTED_MANIFEST_SHA256 or not sidecar_ok(NESTED_MANIFEST):
    raise RuntimeError("Nested-SCV manifest verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256 or not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4H0 manifest verification failed.")

eval_metadata = pq.ParquetFile(EVAL).metadata
nested_metadata = pq.ParquetFile(NESTED_PACKAGE).metadata

if (eval_metadata.num_rows, eval_metadata.num_columns) != (
    EXPECTED["total_rows"],
    EXPECTED["eval_columns"],
):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: "
        f"{(eval_metadata.num_rows, eval_metadata.num_columns)}"
    )

if (nested_metadata.num_rows, nested_metadata.num_columns) != (
    EXPECTED["total_rows"],
    EXPECTED["nested_columns"],
):
    raise RuntimeError(
        f"Unexpected nested-SCV dimensions: "
        f"{(nested_metadata.num_rows, nested_metadata.num_columns)}"
    )

nested_manifest_readback = json.loads(NESTED_MANIFEST.read_text(encoding="utf-8"))
prior_manifest_readback = json.loads(PRIOR_MANIFEST.read_text(encoding="utf-8"))

if prior_manifest_readback.get("decision") != (
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
):
    raise RuntimeError("Prior Cell 6C-4H0 decision mismatch.")


# --------------------------------------------------------------------------------------------------
# 4. VERIFY THE FROZEN NESTED OUTCOMES BEFORE LOADING ANY SCORE
# --------------------------------------------------------------------------------------------------

nested_columns = [
    "t0_row_order",
    "rcv_accession",
    "new_contradictory_high_rigor_status",
    "new_contradictory_high_rigor_submission",
    "distribution_status",
    "submitter_classification_tvd",
    "major_submitter_distribution_shift",
    "policy_version",
    "policy_sha256",
]

nested = pd.read_parquet(NESTED_PACKAGE, columns=nested_columns).copy()
nested["rcv_accession"] = normalize_rcv(nested["rcv_accession"])
nested["t0_row_order"] = pd.to_numeric(
    nested["t0_row_order"],
    errors="raise",
).astype("int64")

if len(nested) != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV row count mismatch.")
if nested["rcv_accession"].isna().any():
    raise RuntimeError("Nested-SCV RCV normalization failure.")
if nested["rcv_accession"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV RCV keys are not unique.")
if nested["t0_row_order"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV row-order keys are not unique.")
if not nested["policy_version"].astype("string").eq("1.1").all():
    raise RuntimeError("Nested-SCV policy-version mismatch.")
if not nested["policy_sha256"].astype("string").eq(EXPECTED_POLICY_SHA256).all():
    raise RuntimeError("Nested-SCV policy-hash mismatch.")

high_rigor = nested["new_contradictory_high_rigor_submission"]
hr_events = int(high_rigor.fillna(0).astype("int8").sum())
hr_evaluable = int(high_rigor.notna().sum())
hr_negatives = hr_evaluable - hr_events
hr_censored = len(nested) - hr_evaluable

if (
    hr_events,
    hr_negatives,
    hr_censored,
) != (
    EXPECTED["high_rigor_events"],
    EXPECTED["high_rigor_negatives"],
    EXPECTED["high_rigor_censored"],
):
    raise RuntimeError(
        "High-rigor contradiction endpoint accounting mismatch: "
        f"{(hr_events, hr_negatives, hr_censored)}"
    )

complete = nested.loc[nested["distribution_status"].eq("EVALUABLE")].copy()
distribution_censored = int(nested["distribution_status"].ne("EVALUABLE").sum())

if len(complete) != EXPECTED["complete_rows"]:
    raise RuntimeError("Complete-case nested-SCV row count mismatch.")
if distribution_censored != EXPECTED["censored"]:
    raise RuntimeError("Nested-SCV distribution censoring count mismatch.")
if complete["major_submitter_distribution_shift"].isna().any():
    raise RuntimeError("Complete-case major-shift outcome contains missing values.")

y_prejoin = complete["major_submitter_distribution_shift"].astype("int8").to_numpy()
if (
    int(y_prejoin.sum()),
    int(len(y_prejoin) - y_prejoin.sum()),
) != (
    EXPECTED["events"],
    EXPECTED["negatives"],
):
    raise RuntimeError("Nested-SCV event accounting mismatch.")

tvd_values = set(
    np.round(
        pd.to_numeric(
            complete["submitter_classification_tvd"],
            errors="raise",
        ).to_numpy(float),
        12,
    ).tolist()
)
if tvd_values != {0.0, 1.0}:
    raise RuntimeError(f"Unexpected complete-case TVD support: {tvd_values}")


# --------------------------------------------------------------------------------------------------
# 5. LOAD FROZEN SCORES ONLY AFTER THE SCORE-BLIND NESTED PACKAGE PASSES
# --------------------------------------------------------------------------------------------------

score_columns = [column for column, _ in SCORES.values()]
required_score_columns = KEYS + score_columns
eval_schema = pq.ParquetFile(EVAL).schema_arrow.names
missing_score_columns = [
    column for column in required_score_columns if column not in eval_schema
]
if missing_score_columns:
    raise RuntimeError(f"Missing Stage 6B score columns: {missing_score_columns}")

scores = pd.read_parquet(EVAL, columns=required_score_columns).copy()
scores["rcv_accession"] = normalize_rcv(scores["rcv_accession"])
scores["t0_row_order"] = pd.to_numeric(
    scores["t0_row_order"],
    errors="raise",
).astype("int64")

if len(scores) != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score row count mismatch.")
if scores["rcv_accession"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score RCV keys are not unique.")
if scores["t0_row_order"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score row-order keys are not unique.")

for column in score_columns:
    scores[column] = pd.to_numeric(scores[column], errors="raise").astype("float64")
    values = scores[column].to_numpy(float)
    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid Stage 6B score field: {column}")

analysis = complete.merge(
    scores,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True,
)
if not analysis["_merge"].eq("both").all():
    raise RuntimeError("A complete-case nested-SCV row failed score linkage.")

analysis = (
    analysis.drop(columns="_merge")
    .sort_values("t0_row_order", kind="mergesort")
    .reset_index(drop=True)
)

if len(analysis) != EXPECTED["complete_rows"]:
    raise RuntimeError("Final 37-record analysis cohort size mismatch.")

y = analysis["major_submitter_distribution_shift"].astype("int8").to_numpy()
prevalence = float(y.mean())

if abs(prevalence - EXPECTED["prevalence"]) > 1e-15:
    raise RuntimeError("37-record endpoint prevalence mismatch.")


# --------------------------------------------------------------------------------------------------
# 6. ENDPOINT ACCOUNTING AND LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

endpoint_accounting = pd.DataFrame([
    {
        "endpoint_key": "new_contradictory_high_rigor_submission",
        "endpoint": "New contradictory high-rigor submission",
        "status": "NOT_ESTIMABLE_NO_POSITIVE_EVENTS",
        "evaluable_rows": hr_events + hr_negatives,
        "events": hr_events,
        "negatives": hr_negatives,
        "censored_rows": hr_censored,
        "censoring_fraction": hr_censored / EXPECTED["total_rows"],
        "analyzed_for_discrimination": False,
        "interpretation": (
            "The frozen nested source provides zero observable positive high-rigor events; "
            "AUPRC and AUROC are not estimable and the outcome is not redefined."
        ),
    },
    {
        "endpoint_key": "major_submitter_distribution_shift",
        "endpoint": "Major submitter-classification distribution shift",
        "status": "TECHNICALLY_ESTIMABLE_HIGHLY_EXPLORATORY",
        "evaluable_rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "censored_rows": EXPECTED["censored"],
        "censoring_fraction": EXPECTED["censoring_fraction"],
        "analyzed_for_discrimination": True,
        "interpretation": (
            "Only 37 complete-case records are evaluable; all inference is descriptive and "
            "highly exploratory."
        ),
    },
])

point_rows = []
for model_key, (score_column, model_name) in SCORES.items():
    score = analysis[score_column].to_numpy(float)
    auprc = float(average_precision_score(y, score))
    auroc = float(roc_auc_score(y, score))
    point_rows.append({
        "endpoint_key": "major_submitter_distribution_shift",
        "model_key": model_key,
        "model": model_name,
        "score_column": score_column,
        "rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "prevalence": prevalence,
        "unique_score_values": int(np.unique(score).size),
        "point_auprc": auprc,
        "auprc_minus_prevalence": auprc - prevalence,
        "auprc_lift_over_prevalence": auprc / prevalence,
        "point_auroc": auroc,
        "auroc_minus_0_50": auroc - 0.50,
        "analysis_status": "HIGHLY_EXPLORATORY_37_COMPLETE_CASE_RECORDS",
    })

point_estimates = pd.DataFrame(point_rows)
point_lookup = point_estimates.set_index("model_key")


# --------------------------------------------------------------------------------------------------
# 7. EXACT CELL 6C-3G1 2,000-ATTEMPT PAIRED ORDINARY ROW BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    f"\nPreparing the frozen 37-record nested-SCV complete-case endpoint: "
    f"{EXPECTED['events']} events and {EXPECTED['negatives']} negatives"
)

rng = np.random.default_rng(SEED)
replicate_rows = []
valid = 0
invalid = 0
analysis_start = time.time()

score_arrays = {
    model_key: analysis[score_column].to_numpy(float)
    for model_key, (score_column, _) in SCORES.items()
}

for attempt in range(1, N_BOOT + 1):
    # Exact historical Cell 6C-3G1 bootstrap implementation.
    indices = rng.integers(0, len(y), size=len(y))
    y_boot = y[indices]
    sampled_events = int(y_boot.sum())
    sampled_negatives = int(len(y_boot) - sampled_events)
    valid_two_class = bool(np.unique(y_boot).size == 2)

    row = {
        "replicate": attempt,
        "sampled_rows": len(y_boot),
        "sampled_events": sampled_events,
        "sampled_negatives": sampled_negatives,
        "valid_two_class_replicate": valid_two_class,
        "seed": SEED,
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
    }

    if valid_two_class:
        valid += 1
        for model_key in SCORES:
            score_boot = score_arrays[model_key][indices]
            row[f"{model_key}__auprc"] = float(
                average_precision_score(y_boot, score_boot)
            )
            row[f"{model_key}__auroc"] = float(
                roc_auc_score(y_boot, score_boot)
            )
    else:
        invalid += 1
        for model_key in SCORES:
            row[f"{model_key}__auprc"] = np.nan
            row[f"{model_key}__auroc"] = np.nan

    replicate_rows.append(row)

    if attempt % 250 == 0:
        print(
            f"  Completed {attempt:,}/{N_BOOT:,} replicates | "
            f"valid {valid:,}"
        )

bootstrap_elapsed = time.time() - analysis_start
bootstrap_replicates = pd.DataFrame(replicate_rows)

if (valid, invalid) != (N_BOOT, 0):
    raise RuntimeError(
        f"Historical Cell 6C-3G1 validity mismatch: valid={valid}, invalid={invalid}"
    )


# --------------------------------------------------------------------------------------------------
# 8. MODEL INTERVALS AND PAIRED EXPLORATORY INFERENCE
# --------------------------------------------------------------------------------------------------

interval_rows = []
for model_key, (_, model_name) in SCORES.items():
    auprc_values = bootstrap_replicates[f"{model_key}__auprc"].to_numpy(float)
    auroc_values = bootstrap_replicates[f"{model_key}__auroc"].to_numpy(float)
    auprc_low, auprc_high = ci(auprc_values)
    auroc_low, auroc_high = ci(auroc_values)
    point = point_lookup.loc[model_key]

    interval_rows.append({
        "endpoint_key": "major_submitter_distribution_shift",
        "model_key": model_key,
        "model": model_name,
        "rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "prevalence": prevalence,
        "unique_score_values": int(point["unique_score_values"]),
        "point_auprc": float(point["point_auprc"]),
        "auprc_ci_lower": auprc_low,
        "auprc_ci_upper": auprc_high,
        "point_auprc_minus_prevalence": float(point["auprc_minus_prevalence"]),
        "point_auroc": float(point["point_auroc"]),
        "auroc_ci_lower": auroc_low,
        "auroc_ci_upper": auroc_high,
        "point_auroc_minus_0_50": float(point["auroc_minus_0_50"]),
        "attempted_bootstrap_replicates": N_BOOT,
        "valid_bootstrap_replicates": valid,
        "invalid_one_class_replicates": invalid,
        "analysis_status": "HIGHLY_EXPLORATORY_37_COMPLETE_CASE_RECORDS",
    })

model_intervals = pd.DataFrame(interval_rows)

paired_rows = []
for comparator_key in PRINCIPAL_COMPARATORS:
    comparator_name = SCORES[comparator_key][1]
    for metric_name, metric_label, point_column in [
        ("auprc", "AUPRC", "point_auprc"),
        ("auroc", "AUROC", "point_auroc"),
    ]:
        differences = (
            bootstrap_replicates[f"full_ges__{metric_name}"].to_numpy(float)
            - bootstrap_replicates[f"{comparator_key}__{metric_name}"].to_numpy(float)
        )
        lower, upper = ci(differences)
        point_difference = float(
            point_lookup.loc["full_ges", point_column]
            - point_lookup.loc[comparator_key, point_column]
        )

        paired_rows.append({
            "endpoint_key": "major_submitter_distribution_shift",
            "metric": metric_label,
            "comparison": f"Full GES minus {comparator_name}",
            "comparator_key": comparator_key,
            "comparator": comparator_name,
            "point_difference": point_difference,
            "difference_ci_lower": lower,
            "difference_ci_upper": upper,
            "paired_interval_status": interval_status(lower, upper),
            "bootstrap_probability_full_greater": float(np.mean(differences > 0)),
            "bootstrap_sign_p_value": sign_p(differences),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
            "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
            "analysis_status": "HIGHLY_EXPLORATORY_NO_MULTIPLICITY_CLAIM",
        })

paired_inference = pd.DataFrame(paired_rows)


# --------------------------------------------------------------------------------------------------
# 9. HISTORICAL-RESULT PRESERVATION AND CONCORDANCE
# --------------------------------------------------------------------------------------------------

historical_metric_values = {
    "full_ges": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "no_star_ges": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "review_stars": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "combined_metadata": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "conflict": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "recency": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "submitter": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "entropy": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "additive": {"AUPRC": 0.567568, "AUROC": 0.500000},
}

historical_rows = [
    {
        "result_id": "endpoint__evaluable_rows",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "evaluable_rows",
        "historical_value": 37.0,
        "historical_conclusion": "37_complete_case_records",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__events",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "events",
        "historical_value": 21.0,
        "historical_conclusion": "21_major_shift_events",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__negatives",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "negatives",
        "historical_value": 16.0,
        "historical_conclusion": "16_major_shift_negatives",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__censored_rows",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "censored_rows",
        "historical_value": 66_599.0,
        "historical_conclusion": "66599_distribution_censored",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__high_rigor_events",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "high_rigor_events",
        "historical_value": 0.0,
        "historical_conclusion": "high_rigor_endpoint_not_estimable",
        "tolerance": 0.0,
    },
    {
        "result_id": "bootstrap__valid_replicates",
        "result_type": "bootstrap_accounting",
        "model_key": "",
        "metric": "valid_replicates",
        "historical_value": 2_000.0,
        "historical_conclusion": "2000_valid_bootstrap_replicates",
        "tolerance": 0.0,
    },
]

for model_key, metric_values in historical_metric_values.items():
    for metric, value in metric_values.items():
        historical_rows.append({
            "result_id": f"{model_key}__{metric.lower()}",
            "result_type": "model_point",
            "model_key": model_key,
            "metric": metric,
            "historical_value": value,
            "historical_conclusion": "historical_rounded_point_reproduced",
            "tolerance": 5.1e-7,
        })

historical_results = pd.DataFrame(historical_rows)
historical_results["source"] = (
    "Technical report Version 7.0 Appendix P and original Cell 6C-3G1; "
    "historical rounded point values are preserved separately from exact reproduced values."
)

concordance_rows = []
for record in historical_results.to_dict("records"):
    result_type = record["result_type"]
    metric = record["metric"]

    if result_type == "endpoint_accounting":
        reproduced_map = {
            "evaluable_rows": float(EXPECTED["complete_rows"]),
            "events": float(EXPECTED["events"]),
            "negatives": float(EXPECTED["negatives"]),
            "censored_rows": float(EXPECTED["censored"]),
            "high_rigor_events": float(hr_events),
        }
        conclusion_map = {
            "evaluable_rows": "37_complete_case_records",
            "events": "21_major_shift_events",
            "negatives": "16_major_shift_negatives",
            "censored_rows": "66599_distribution_censored",
            "high_rigor_events": "high_rigor_endpoint_not_estimable",
        }
        reproduced_value = reproduced_map[metric]
        reproduced_conclusion = conclusion_map[metric]

    elif result_type == "bootstrap_accounting":
        reproduced_value = float(valid)
        reproduced_conclusion = "2000_valid_bootstrap_replicates"

    elif result_type == "model_point":
        model_key = record["model_key"]
        column = "point_auprc" if metric == "AUPRC" else "point_auroc"
        reproduced_value = float(point_lookup.loc[model_key, column])
        reproduced_conclusion = "historical_rounded_point_reproduced"

    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    absolute_difference = abs(reproduced_value - float(record["historical_value"]))
    point_pass = absolute_difference <= float(record["tolerance"])
    conclusion_pass = reproduced_conclusion == record["historical_conclusion"]

    concordance_rows.append({
        **record,
        "reproduced_value": reproduced_value,
        "absolute_difference": absolute_difference,
        "value_reproduced_at_recorded_precision": point_pass,
        "reproduced_conclusion": reproduced_conclusion,
        "scientific_conclusion_concordant": conclusion_pass,
    })

concordance = pd.DataFrame(concordance_rows)

limitations = pd.DataFrame([
    {
        "limitation_key": "complete_case_size",
        "observed_value": "37 evaluable records",
        "interpretation": (
            "The endpoint is technically estimable but too small to establish generalizable "
            "superiority, calibration, or clinical utility."
        ),
    },
    {
        "limitation_key": "censoring",
        "observed_value": f"{EXPECTED['censored']:,}/{EXPECTED['total_rows']:,} censored",
        "interpretation": (
            "The 99.9445% censoring fraction is the dominant scientific limitation and must be "
            "reported with every result."
        ),
    },
    {
        "limitation_key": "high_rigor_endpoint",
        "observed_value": "0 positive events",
        "interpretation": (
            "The high-rigor contradictory-submission endpoint is not estimable and is not "
            "redefined using aggregate review metadata."
        ),
    },
    {
        "limitation_key": "tvd_support",
        "observed_value": "Observed TVD values are only 0 and 1",
        "interpretation": (
            "Continuous TVD analysis is not treated as a stable continuous endpoint in this tiny "
            "complete-case subset."
        ),
    },
    {
        "limitation_key": "multiplicity_and_inference",
        "observed_value": "Exploratory paired bootstrap only",
        "interpretation": (
            "Intervals describe resampling instability inside the 37-record subset and do not "
            "support confirmatory multiplicity-adjusted claims."
        ),
    },
])


# --------------------------------------------------------------------------------------------------
# 10. FRESH QC BEFORE ARTIFACT WRITES
# --------------------------------------------------------------------------------------------------

checks = []

def check(name: str, passed: bool, details):
    checks.append({
        "check_name": name,
        "passed": bool(passed),
        "details": native(details),
    })

dynamic_models = ["full_ges", "no_star_ges", "combined_metadata", "recency"]
constant_models = ["review_stars", "conflict", "submitter", "entropy", "additive"]

check("stage6b_hash", sha(EVAL) == EVAL_SHA256, sha(EVAL))
check("stage6b_sidecar", sidecar_ok(EVAL), str(EVAL) + ".sha256")
check("nested_package_hash", sha(NESTED_PACKAGE) == NESTED_PACKAGE_SHA256, sha(NESTED_PACKAGE))
check("nested_package_sidecar", sidecar_ok(NESTED_PACKAGE), str(NESTED_PACKAGE) + ".sha256")
check("nested_manifest_hash", sha(NESTED_MANIFEST) == NESTED_MANIFEST_SHA256, sha(NESTED_MANIFEST))
check("nested_manifest_sidecar", sidecar_ok(NESTED_MANIFEST), str(NESTED_MANIFEST) + ".sha256")
check("prior_4h0_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256, sha(PRIOR_MANIFEST))
check("prior_4h0_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST) + ".sha256")
check("stage6b_dimensions", (eval_metadata.num_rows, eval_metadata.num_columns) == (66_636, 79), {})
check("nested_dimensions", (nested_metadata.num_rows, nested_metadata.num_columns) == (66_636, 28), {})
check("nested_policy", nested["policy_sha256"].astype("string").eq(EXPECTED_POLICY_SHA256).all(), {})
check("high_rigor_nonestimable", (hr_events, hr_negatives, hr_censored) == (0, 54_228, 12_408), {})
check("distribution_accounting", (len(analysis), int(y.sum()), int(len(y)-y.sum()), distribution_censored) == (37, 21, 16, 66_599), {})
check(
    "distribution_censoring_fraction",
    abs(
        EXPECTED["censoring_fraction"]
        - (EXPECTED["censored"] / EXPECTED["total_rows"])
    ) < 1e-15,
    {
        "observed": EXPECTED["censoring_fraction"],
        "expected_from_counts": EXPECTED["censored"] / EXPECTED["total_rows"],
    },
)
check("tvd_support", tvd_values == {0.0, 1.0}, sorted(tvd_values))
check("score_join_complete", len(analysis) == 37 and analysis[score_columns].notna().all().all(), {})
check("nine_point_estimates", len(point_estimates) == 9, len(point_estimates))
check("bootstrap_attempts", len(bootstrap_replicates) == 2_000, len(bootstrap_replicates))
check("bootstrap_validity", (valid, invalid) == (2_000, 0), {"valid": valid, "invalid": invalid})
check("nine_model_intervals", len(model_intervals) == 9, len(model_intervals))
check("six_paired_results", len(paired_inference) == 6, len(paired_inference))
check(
    "dynamic_model_historical_metrics",
    all(
        abs(point_lookup.loc[key, "point_auprc"] - 0.660749) <= 5.1e-7
        and abs(point_lookup.loc[key, "point_auroc"] - 0.559524) <= 5.1e-7
        for key in dynamic_models
    ),
    point_estimates.loc[
        point_estimates["model_key"].isin(dynamic_models),
        ["model_key", "point_auprc", "point_auroc"],
    ].to_dict("records"),
)
check(
    "constant_comparator_metrics",
    all(
        int(point_lookup.loc[key, "unique_score_values"]) == 1
        and abs(point_lookup.loc[key, "point_auprc"] - prevalence) <= 1e-15
        and abs(point_lookup.loc[key, "point_auroc"] - 0.5) <= 1e-15
        for key in constant_models
    ),
    point_estimates.loc[
        point_estimates["model_key"].isin(constant_models),
        ["model_key", "unique_score_values", "point_auprc", "point_auroc"],
    ].to_dict("records"),
)
check(
    "full_no_star_identical_metrics",
    abs(point_lookup.loc["full_ges", "point_auprc"] - point_lookup.loc["no_star_ges", "point_auprc"]) <= 1e-15
    and abs(point_lookup.loc["full_ges", "point_auroc"] - point_lookup.loc["no_star_ges", "point_auroc"]) <= 1e-15,
    {},
)
check(
    "full_combined_identical_metrics",
    abs(point_lookup.loc["full_ges", "point_auprc"] - point_lookup.loc["combined_metadata", "point_auprc"]) <= 1e-15
    and abs(point_lookup.loc["full_ges", "point_auroc"] - point_lookup.loc["combined_metadata", "point_auroc"]) <= 1e-15,
    {},
)
check(
    "historical_values",
    concordance["value_reproduced_at_recorded_precision"].all(),
    concordance.loc[
        ~concordance["value_reproduced_at_recorded_precision"],
        ["result_id", "historical_value", "reproduced_value", "absolute_difference", "tolerance"],
    ].to_dict("records"),
)
check(
    "historical_conclusions",
    concordance["scientific_conclusion_concordant"].all(),
    concordance.loc[
        ~concordance["scientific_conclusion_concordant"],
        ["result_id", "historical_conclusion", "reproduced_conclusion"],
    ].to_dict("records"),
)
check("limitations_complete", len(limitations) == 5, len(limitations))
check(
    "frozen_sources_unchanged",
    sha(EVAL) == EVAL_SHA256
    and sha(NESTED_PACKAGE) == NESTED_PACKAGE_SHA256
    and sha(NESTED_MANIFEST) == NESTED_MANIFEST_SHA256
    and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256,
    {},
)

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError(
        "QC failed before writing:\n"
        + json.dumps(native(failed), indent=2)
    )


# --------------------------------------------------------------------------------------------------
# 11. VERSIONED WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

analysis_artifact_columns = (
    KEYS
    + [
        "submitter_classification_tvd",
        "major_submitter_distribution_shift",
        "policy_version",
        "policy_sha256",
    ]
    + score_columns
)
analysis_artifact = analysis[analysis_artifact_columns].copy()

write_csv(P["endpoint_accounting"], endpoint_accounting)
write_parquet(P["analysis_cohort"], analysis_artifact)
write_csv(P["point_estimates"], point_estimates)
write_parquet(P["bootstrap_replicates"], bootstrap_replicates)
write_csv(P["model_intervals"], model_intervals)
write_csv(P["paired_inference"], paired_inference)
write_csv(P["historical_results"], historical_results)
write_csv(P["concordance"], concordance)
write_csv(P["limitations"], limitations)

readback = {
    "endpoint_accounting": len(pd.read_csv(P["endpoint_accounting"])) == 2,
    "analysis_cohort": len(pd.read_parquet(P["analysis_cohort"])) == 37,
    "point_estimates": len(pd.read_csv(P["point_estimates"])) == 9,
    "bootstrap_replicates": len(pd.read_parquet(P["bootstrap_replicates"])) == 2_000,
    "model_intervals": len(pd.read_csv(P["model_intervals"])) == 9,
    "paired_inference": len(pd.read_csv(P["paired_inference"])) == 6,
    "historical_results": len(pd.read_csv(P["historical_results"])) == len(historical_results),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
    "limitations": len(pd.read_csv(P["limitations"])) == 5,
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4I0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "nested_scv_37_record_exploratory_materialization",
    "immutable_sources": {
        "stage6b_evaluable": {"path": str(EVAL), "sha256": sha(EVAL)},
        "nested_scv_record_package": {
            "path": str(NESTED_PACKAGE),
            "sha256": sha(NESTED_PACKAGE),
        },
        "nested_scv_manifest": {
            "path": str(NESTED_MANIFEST),
            "sha256": sha(NESTED_MANIFEST),
        },
        "prior_6c4h0_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
        },
    },
    "bootstrap": {
        "method": "paired ordinary row bootstrap with replacement",
        "exact_historical_implementation": "rng.integers(0, 37, size=37)",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "valid_replicates": valid,
        "invalid_one_class_replicates": invalid,
        "identical_resamples_across_nine_scores": True,
    },
    "endpoint_accounting": endpoint_accounting.to_dict("records"),
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "endpoint_accounting",
    "analysis_cohort",
    "point_estimates",
    "bootstrap_replicates",
    "model_intervals",
    "paired_inference",
    "historical_results",
    "concordance",
    "limitations",
    "qc",
]

for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }

    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        record.update(rows=metadata.num_rows, columns=metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True

    return record


manifest = {
    "cell_id": "6C-4I0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "nested_scv_37_record_exploratory_analysis_materialization",
    "immutable_sources": qc_payload["immutable_sources"],
    "analysis_lock": {
        "endpoint": "major_submitter_distribution_shift",
        "threshold": "submitter_classification_tvd >= 0.50",
        "complete_case_rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "censored_rows": EXPECTED["censored"],
        "scores": {
            model_key: {"column": column, "display_name": display_name}
            for model_key, (column, display_name) in SCORES.items()
        },
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_implementation": "rng.integers(0, 37, size=37)",
        "principal_comparators": PRINCIPAL_COMPARATORS,
    },
    "result_summary": {
        "prevalence": prevalence,
        "censoring_fraction": EXPECTED["censoring_fraction"],
        "full_ges_auprc": float(point_lookup.loc["full_ges", "point_auprc"]),
        "full_ges_auroc": float(point_lookup.loc["full_ges", "point_auroc"]),
        "no_star_ges_auprc": float(point_lookup.loc["no_star_ges", "point_auprc"]),
        "no_star_ges_auroc": float(point_lookup.loc["no_star_ges", "point_auroc"]),
        "combined_metadata_auprc": float(
            point_lookup.loc["combined_metadata", "point_auprc"]
        ),
        "combined_metadata_auroc": float(
            point_lookup.loc["combined_metadata", "point_auroc"]
        ),
        "recency_auprc": float(point_lookup.loc["recency", "point_auprc"]),
        "recency_auroc": float(point_lookup.loc["recency", "point_auroc"]),
        "historical_values_reproduced": bool(
            concordance["value_reproduced_at_recorded_precision"].all()
        ),
        "historical_conclusions_concordant": bool(
            concordance["scientific_conclusion_concordant"].all()
        ),
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "high_rigor_endpoint_analyzed": False,
        "high_rigor_endpoint_reason": "zero_observable_positive_events",
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_definition_changed": False,
        "policy_changed": False,
        "linkage_decisions_changed": False,
        "row_order_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "The 37-record endpoint is technically estimable but highly exploratory. Full GES, "
            "no-star GES, combined metadata, and recency have identical descriptive AUPRC/AUROC. "
            "The remaining low-cardinality comparators are constant. With 99.9445% censoring, "
            "these results cannot establish generalizable superiority, calibration, clinical "
            "utility, or RAG safety."
        ),
    },
    "decision": (
        "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_category": "leave_one_gene_out_validation_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Fresh package and immutable-source reverification.
manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))

if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")

for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")

if (
    sha(EVAL) != EVAL_SHA256
    or sha(NESTED_PACKAGE) != NESTED_PACKAGE_SHA256
    or sha(NESTED_MANIFEST) != NESTED_MANIFEST_SHA256
    or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256
):
    raise RuntimeError("An immutable source changed during Cell 6C-4I0.")


# --------------------------------------------------------------------------------------------------
# 12. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4I — CELL 6C-4I0 — "
    "37-RECORD NESTED-SCV EXPLORATORY RESULT CATEGORY"
)
print("=" * 170)
print(f"Stage 6B evaluable hash                 : PASS ({sha(EVAL)})")
print(f"Nested-SCV package hash                 : PASS ({sha(NESTED_PACKAGE)})")
print(f"Nested-SCV manifest hash                : PASS ({sha(NESTED_MANIFEST)})")
print(f"Prior Cell 6C-4H0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    "High-rigor contradiction endpoint      : NOT ESTIMABLE "
    f"({hr_events} events, {hr_negatives:,} negatives, {hr_censored:,} censored)"
)
print(
    f"Distribution endpoint                   : {EXPECTED['complete_rows']} rows | "
    f"{EXPECTED['events']} events | {EXPECTED['negatives']} negatives"
)
print(
    f"Distribution censoring                  : {EXPECTED['censored']:,}/"
    f"{EXPECTED['total_rows']:,} ({100 * EXPECTED['censoring_fraction']:.4f}%)"
)
print(f"Nine-score point estimates              : PASS ({len(point_estimates)}/9)")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(f"Valid / invalid replicates              : {valid:,} / {invalid:,}")
print(f"Bootstrap elapsed                       : {bootstrap_elapsed / 60:.2f} minutes")
print(f"Model intervals                         : PASS ({len(model_intervals)}/9)")
print(f"Principal paired comparisons            : PASS ({len(paired_inference)}/6)")
print(
    f"Historical recorded values              : PASS "
    f"({int(concordance.value_reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(
    f"Historical scientific conclusions       : PASS "
    f"({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})"
)
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")

for label, key in [
    ("Endpoint accounting", "endpoint_accounting"),
    ("37-record analysis cohort", "analysis_cohort"),
    ("Point estimates", "point_estimates"),
    ("Bootstrap replicates", "bootstrap_replicates"),
    ("Model intervals", "model_intervals"),
    ("Paired exploratory inference", "paired_inference"),
    ("Historical results", "historical_results"),
    ("Concordance table", "concordance"),
    ("Limitations", "limitations"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")

print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nNESTED-SCV ENDPOINT ACCOUNTING")
print(endpoint_accounting.to_string(index=False))

print("\n37-RECORD NINE-SCORE MODEL INTERVALS")
print(
    model_intervals[
        [
            "model",
            "unique_score_values",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "point_auprc_minus_prevalence",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "point_auroc_minus_0_50",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ].to_string(index=False)
)

print("\nFULL-GES PRINCIPAL-COMPARATOR PAIRED EXPLORATORY INFERENCE")
print(paired_inference.to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print(
    "Only 37 of 66,636 records are evaluable and 99.9445% are censored. "
    "Full GES, no-star GES, combined metadata, and recency have identical descriptive "
    "discrimination in this tiny subset. The high-rigor contradiction endpoint has zero "
    "positive events and was not analyzed. These estimates do not establish generalizable "
    "superiority, calibration, clinical utility, or RAG safety."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The seventh of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is leave-one-gene-out validation materialization. "
    "Experiment 2 has not started."
)
print("=" * 170)


Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4I0_Nested_SCV_37_Record_Exploratory_Materialization.ipynb

Preparing the frozen 37-record nested-SCV complete-case endpoint: 21 events and 16 negatives
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicates | valid 1,000
  Completed 1,250/2,000 replicates | valid 1,250
  Completed 1,500/2,000 replicates | valid 1,500
  Completed 1,750/2,000 replicates | valid 1,750
  Completed 2,000/2,000 replicates | valid 2,000

STAGE 6C STEP 4I — CELL 6C-4I0 — 37-RECORD NESTED-SCV EXPLORATORY RESULT CATEGORY
Stage 6B evaluable hash                 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Nested-SCV package hash                 : PASS (18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2)
Nested-SCV manifest hash                : PASS (b89626648af773b79053515f81d